<a href="https://colab.research.google.com/github/pujisubarkah/replikasi_DAiSEE-Towards-User-Engagement-Recognition-in-the-Wild/blob/main/Notebook_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

DATA LOADER

In [ ]:
import os
import cv2
import torch
import numpy as np
import pandas as pd

from pathlib import Path
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

import kagglehub

In [ ]:
dataset_path = kagglehub.dataset_download(
    "olgaparfenova/daisee"
)

DAISEE_PATH = os.path.join(dataset_path,"DAiSEE")

DATA_PATH = os.path.join(DAISEE_PATH,"DataSet")

LABEL_PATH = os.path.join(DAISEE_PATH,"Labels")

TRAIN_PATH = os.path.join(DATA_PATH,"Train")
VAL_PATH   = os.path.join(DATA_PATH,"Validation")
TEST_PATH  = os.path.join(DATA_PATH,"Test")

Using Colab cache for faster access to the 'daisee' dataset.


In [ ]:
train_df = pd.read_csv(
    os.path.join(LABEL_PATH,"TrainLabels.csv")
)

val_df = pd.read_csv(
    os.path.join(LABEL_PATH,"ValidationLabels.csv")
)

test_df = pd.read_csv(
    os.path.join(LABEL_PATH,"TestLabels.csv")
)

In [ ]:
train_df.head()

,ClipID,Boredom,Engagement,Confusion,Frustration
0,1100011002.avi,0,2,0,0
1,1100011003.avi,0,2,0,0
2,1100011004.avi,0,3,0,0
3,1100011005.avi,0,3,0,0
4,1100011006.avi,0,3,0,0


In [ ]:
#load video
def load_video(video_path):

    cap = cv2.VideoCapture(video_path)

    frames = []

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        frames.append(frame)

    cap.release()

    return frames

In [ ]:
sample_clip_id = train_df.iloc[0]['ClipID'] # Get the first ClipID from train_df
subject = sample_clip_id[:6] # Extract subject from ClipID

# Construct the video_path based on the expected directory structure
video_path = os.path.join(TRAIN_PATH, subject, sample_clip_id.replace('.avi', ''), sample_clip_id)

frames = load_video(video_path)

print("Jumlah frame:", len(frames))

Jumlah frame: 300


In [ ]:
def uniform_sampling(frames, num_frames=30):

    indices = np.linspace(
        0,
        len(frames)-1,
        num_frames,
        dtype=int
    )

    sampled = [frames[i] for i in indices]

    return sampled

In [ ]:
sampled_frames = uniform_sampling(frames)

print(len(sampled_frames))

30


In [ ]:
def resize_frames(frames, size=(224,224)):

    return [
        cv2.resize(frame, size)
        for frame in frames
    ]

In [ ]:
def normalize_frames(frames):

    frames = np.array(frames, dtype=np.float32)

    frames /= 255.0

    return frames

In [ ]:
def preprocess_video(video_path):

    frames = load_video(video_path)

    frames = uniform_sampling(frames)

    frames = resize_frames(frames)

    frames = normalize_frames(frames)

    return frames

In [ ]:
def get_video_path(split_path, clip_name):

    clip_id = os.path.splitext(clip_name)[0]

    subject = clip_id[:6]

    video_path = os.path.join(
        split_path,
        subject,
        clip_id,
        clip_name
    )

    return video_path

In [ ]:
clip = train_df.iloc[0]["ClipID"]

video_path = get_video_path(
    TRAIN_PATH,
    clip
)

print(video_path)

print(os.path.exists(video_path))

/kaggle/input/daisee/DAiSEE/DataSet/Train/110001/1100011002/1100011002.avi
True


In [ ]:
sample = train_df[
    train_df["ClipID"].str.endswith(".mp4")
].iloc[0]

clip = sample["ClipID"]

video_path = get_video_path(
    TRAIN_PATH,
    clip
)

print(video_path)

print(os.path.exists(video_path))

/kaggle/input/daisee/DAiSEE/DataSet/Train/202614/2026140111/2026140111.mp4
True


In [ ]:
#validasi seluruh dataset
missing=[]

for clip in train_df["ClipID"]:

    path=get_video_path(
        TRAIN_PATH,
        clip
    )

    if not os.path.exists(path):

        missing.append(clip)

print("Missing :",len(missing))

Missing : 0


In [ ]:
class DAiSEEDataset(Dataset):

    def __init__(self,
                 dataframe,
                 split_path,
                 label_column="Engagement"):

        self.df = dataframe.reset_index(drop=True)
        self.split_path = split_path
        self.label_column = label_column

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        clip_name = row["ClipID"]
        label = row[self.label_column]

        video_path = get_video_path(
            self.split_path,
            clip_name
        )

        video = preprocess_video(video_path)

        # (30,224,224,3)
        # menjadi
        # (30,3,224,224)

        video = np.transpose(
            video,
            (0,3,1,2)
        )

        video = torch.tensor(
            video,
            dtype=torch.float32
        )

        label = torch.tensor(
            label,
            dtype=torch.long
        )

        return video, label

In [ ]:
train_dataset = DAiSEEDataset(
    train_df,
    TRAIN_PATH
)

print(len(train_dataset))

5358


In [ ]:
video, label = train_dataset[0]

print(video.shape)
print(label)

torch.Size([30, 3, 224, 224])
tensor(2)


In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0
)

In [ ]:
videos, labels = next(iter(train_loader))

print(videos.shape)
print(labels.shape)

torch.Size([8, 30, 3, 224, 224])
torch.Size([8])


## Kesimpulan Progres Saat Ini

Kita telah berhasil menyiapkan pipeline untuk memuat dan memproses data video dari dataset DAiSEE. Berikut adalah poin-poin penting yang telah tercapai:

1.  **Pengaturan Jalur Data**: Dataset DAiSEE telah berhasil diunduh dan semua jalur yang relevan (untuk data pelatihan, validasi, pengujian, dan label) telah diinisialisasi.
2.  **Pemuatan Data Label**: Data label dari file CSV (TrainLabels.csv, ValidationLabels.csv, TestLabels.csv) telah berhasil dimuat ke dalam DataFrame pandas (`train_df`, `val_df`, `test_df`). `train_df` diketahui memiliki **5358** entri.
3.  **Fungsi Pra-pemrosesan Video**: Serangkaian fungsi penting untuk pra-pemrosesan video telah dibuat dan divalidasi:
    *   `load_video`: Berfungsi untuk mengekstrak bingkai dari file video.
    *   `uniform_sampling`: Untuk mengambil **30 bingkai** secara seragam dari setiap video.
    *   `resize_frames`: Mengubah ukuran setiap bingkai menjadi **(224, 224) piksel**.
    *   `normalize_frames`: Menormalisasi nilai piksel bingkai ke rentang 0-1.
    *   `preprocess_video`: Menggabungkan semua langkah di atas menjadi satu alur pra-pemrosesan.
    *   `get_video_path`: Sebuah fungsi pembantu untuk mendapatkan jalur file video yang benar berdasarkan `ClipID` dan jalur split.
4.  **Validasi Ketersediaan File Video**: Setelah serangkaian pengujian, termasuk pengecekan menyeluruh untuk semua `ClipID` di `train_df`, tidak ada file video yang hilang (`Missing : 0`). Ini mengkonfirmasi bahwa setiap entri dalam `train_df` sekarang memiliki file video yang sesuai dan dapat diakses.
5.  **Pembuatan Dataset dan DataLoader**: Kelas `DAiSEEDataset` kustom telah berhasil dibuat untuk mengelola pemuatan data video dan label. Selanjutnya, `DataLoader` telah dikonfigurasi untuk menyediakan data dalam batch. Contoh batch pertama menunjukkan bentuk tensor video adalah `torch.Size([8, 30, 3, 224, 224])`, mengindikasikan 8 video per batch, masing-masing dengan 30 bingkai, 3 saluran warna, dan dimensi 224x224.

Dengan demikian, infrastruktur dasar untuk memuat, mempra-proses, dan mengelola data video untuk pelatihan model pembelajaran mendalam telah berhasil disiapkan dan diuji.